# WUI Pressure Index Analysis for Tribal Lands

**Series:** Tribal Fire Science & Indigenous Data Sovereignty  
**Author:** Lilly Jones, PhD  
**Last Updated:** 2025  
**Data Sources:** Census TIGER AIANNH, Census Urban Areas, OSM (road density), NLCD (optional)

---

## Overview

This notebook reframes WUI fire risk as **structural**, not just ecological.
Traditional WUI analysis focuses only on distance to cities. This approach captures
the full structural fire risk landscape through five weighted components.

Fast fires = IGNITION + ACCESS + FUELS. 90% of wildfires are human-caused.
Roads are ignition vectors. Urban expansion is increasing exposure over time.
These are factors outside Tribal control — this index quantifies that pressure.

## Composite WUI Pressure Index (0–100)

| Component | Weight | Data Source |
|---|---|---|
| Proximity Pressure | 25% | Census Urban Areas (distance to nearest) |
| Expansion Pressure | 30% | NLCD Impervious Surface change — **manual download** |
| Access Pressure | 20% | OSM road density via osmnx |
| Population Pressure | 15% | Census ACS — **API key required** |
| Interface Complexity | 10% | Edge-to-area ratio from Census TIGER geometry |

Components with missing data are excluded and remaining weights renormalized.

## Data Sovereignty Note
> Tribal boundary data comes from Census TIGER AIANNH. Urban area and road
> data reflect the structural development pressure surrounding Tribal lands —
> pressure that Tribal Nations do not control and did not create.
> All data is real; no synthetic data is used.

In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import os
import warnings
from datetime import datetime

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from shapely.validation import make_valid

from src.data import constants, loaders, validators
from src.data.constants import PRIMARY_TRIBES
from src.geo import utils as geo_utils
from src.indigenous.sovereignty import generate_citations, print_data_acknowledgment
from src.viz import charts, styles

styles.apply_mpl_style()
%matplotlib inline

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

print(f"Repo root : {REPO_ROOT}")
print(f"Output dir: {constants.OUTPUTS_DIR}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
print_data_acknowledgment(
    source_keys=["census_aiannh", "census_urban_areas", "nlcd", "census_acs"]
)

---
## 1. Configuration

In [ ]:
# ── WUI Pressure framework ─────────────────────────────────────────────────────
WUI_COMPONENTS = {
    "proximity":   {"name": "Proximity Pressure",   "weight": 0.25,
                    "description": "Distance to nearest Census Urban Area"},
    "expansion":   {"name": "Expansion Pressure",   "weight": 0.30,
                    "description": "NLCD impervious surface growth (2001→2021)"},
    "access":      {"name": "Access Pressure",      "weight": 0.20,
                    "description": "Road density (ignition vectors) from OSM"},
    "population":  {"name": "Population Pressure",  "weight": 0.15,
                    "description": "Population growth from Census ACS"},
    "complexity":  {"name": "Interface Complexity",  "weight": 0.10,
                    "description": "Edge-to-area ratio from Tribal boundary geometry"},
}

BUFFER_DISTANCES_KM = [5, 10, 20]

# ── Optional data configuration ────────────────────────────────────────────────
# NLCD: Download manually from https://www.mrlc.gov/data
# Place two files in data/raw/nlcd/:
#   nlcd_2001_impervious_conus.img  (or .tif)
#   nlcd_2021_impervious_conus.img
NLCD_2001_PATH = constants.RAW_DIR / "nlcd" / "nlcd_2001_impervious_conus.img"
NLCD_2021_PATH = constants.RAW_DIR / "nlcd" / "nlcd_2021_impervious_conus.img"
NLCD_AVAILABLE = NLCD_2001_PATH.exists() and NLCD_2021_PATH.exists()

# Census ACS API key — free from https://api.census.gov/data/key_signup.html
# Store in .env as CENSUS_API_KEY, never commit.
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", None)
ACS_AVAILABLE  = CENSUS_API_KEY is not None

print("WUI PRESSURE INDEX FRAMEWORK")
print("=" * 60)
for comp_id, comp in WUI_COMPONENTS.items():
    print(f"  {comp['name']} ({comp['weight']:.0%}): {comp['description']}")
print(f"\nBuffer zones: {BUFFER_DISTANCES_KM} km")
print(f"\nOptional data:")
print(f"  NLCD impervious surface : {'AVAILABLE' if NLCD_AVAILABLE else 'NOT FOUND — expansion component will be skipped'}")
print(f"  Census ACS API key      : {'SET' if ACS_AVAILABLE else 'NOT SET — population component will be skipped'}")

---
## 2. Load Data

In [ ]:
# ── Tribal land boundaries ────────────────────────────────────────────────────
all_tribal = loaders.load_census_aian()
all_tribal = validators.validate_geodataframe(
    all_tribal, "census_aiannh", required_columns=["geometry", "NAME"]
)

tribal_lands = all_tribal[all_tribal["NAME"].isin(PRIMARY_TRIBES)].copy()
tribal_lands = tribal_lands.dissolve(by="NAME", as_index=False).reset_index(drop=True)
tribal_lands["geometry"] = tribal_lands.geometry.apply(
    lambda g: make_valid(g) if g is not None else g
)
CONUS = geo_utils.bbox_geodataframe((-127, 24, -65, 50)).geometry.iloc[0]
tribal_lands = tribal_lands[
    tribal_lands.geometry.notnull() &
    tribal_lands.geometry.is_valid &
    tribal_lands.intersects(CONUS)
].copy().reset_index(drop=True)

# Compute geometry metrics in projected CRS
tribal_proj = tribal_lands.to_crs("EPSG:5070")
tribal_lands["area_km2"]      = tribal_proj.geometry.area / 1e6
tribal_lands["perimeter_km"]  = tribal_proj.geometry.length / 1000
tribal_lands["edge_ratio"]    = tribal_lands["perimeter_km"] / tribal_lands["area_km2"]

# Projected centroids for distance calculations
centroids_proj = tribal_proj.geometry.centroid
centroids_geo  = centroids_proj.to_crs(constants.CRS_GEOGRAPHIC)
tribal_lands["centroid_lon"] = centroids_geo.x
tribal_lands["centroid_lat"] = centroids_geo.y

print(f"Tribal land units: {len(tribal_lands)}")
tribal_lands[["NAME", "area_km2", "edge_ratio"]].round(2)

In [ ]:
# ── Census Urban Areas ────────────────────────────────────────────────────────
# Downloads all CONUS urban areas; filter to Urbanized Areas (type=U) only
# to focus on major urban cores as WUI pressure sources.

all_urban = loaders.load_census_urban_areas()
all_urban = validators.validate_geodataframe(
    all_urban, "census_urban_areas",
    required_columns=["geometry", "NAME20"],
)

# Keep only Urbanized Areas (U) — clusters (C) are too small to drive WUI pressure
if "UATYPE20" in all_urban.columns:
    urban_areas = all_urban[all_urban["UATYPE20"] == "U"].copy()
else:
    urban_areas = all_urban.copy()

# Compute projected centroids
urban_proj = urban_areas.to_crs("EPSG:5070")
urban_centroids = urban_proj.geometry.centroid.to_crs(constants.CRS_GEOGRAPHIC)
urban_areas["centroid_lon"] = urban_centroids.x
urban_areas["centroid_lat"] = urban_centroids.y

print(f"Census Urbanized Areas loaded: {len(urban_areas):,}")
urban_areas[["NAME20"]].head()

---
## 3. Compute WUI Pressure Components

In [ ]:
# ── Component 1: Proximity (distance to nearest Urbanized Area) ───────────────
tribal_proj  = tribal_lands.to_crs("EPSG:5070")
urban_proj   = urban_areas.to_crs("EPSG:5070")

nearest_city = []
for _, tribe in tribal_proj.iterrows():
    dists = urban_proj.geometry.centroid.distance(tribe.geometry.centroid)
    idx   = dists.idxmin()
    nearest_city.append({
        "NAME":              tribe["NAME"],
        "nearest_urban":     urban_areas.loc[idx, "NAME20"],
        "distance_to_urban_km": dists.min() / 1000,
    })

proximity_df = pd.DataFrame(nearest_city)
print("Proximity component computed:")
print(proximity_df[["NAME", "nearest_urban", "distance_to_urban_km"]].round(1).to_string(index=False))

In [ ]:
# ── Component 2: Expansion (NLCD impervious surface change) ──────────────────
# Requires manual NLCD download — see configuration in Section 1.

if NLCD_AVAILABLE:
    try:
        import rioxarray as rxr
        import xarray as xr

        expansion_records = []
        for _, tribe in tribal_proj.iterrows():
            bounds = tribe.geometry.bounds
            bbox_5070 = (bounds[0]-5000, bounds[1]-5000, bounds[2]+5000, bounds[3]+5000)

            imp_2001 = rxr.open_rasterio(NLCD_2001_PATH, masked=True).squeeze()
            imp_2021 = rxr.open_rasterio(NLCD_2021_PATH, masked=True).squeeze()

            imp_2001_clip = imp_2001.rio.clip_box(*bbox_5070)
            imp_2021_clip = imp_2021.rio.clip_box(*bbox_5070)

            mean_2001 = float(imp_2001_clip.mean())
            mean_2021 = float(imp_2021_clip.mean())
            change    = mean_2021 - mean_2001

            expansion_records.append({
                "NAME":               tribe["NAME"],
                "impervious_2001":    round(mean_2001, 2),
                "impervious_2021":    round(mean_2021, 2),
                "impervious_change":  round(change, 2),
            })

        expansion_df = pd.DataFrame(expansion_records)
        print("NLCD expansion component computed.")
        print(expansion_df.to_string(index=False))

    except Exception as e:
        print(f"NLCD loading failed: {e}")
        print("Expansion component will be excluded from composite index.")
        expansion_df = None
        NLCD_AVAILABLE = False
else:
    expansion_df = None
    print(
        "NLCD files not found. Expansion component excluded from composite index.\n"
        "Download from https://www.mrlc.gov/data and place in data/raw/nlcd/"
    )

In [ ]:
# ── Component 3: Access (road density from OSM) ───────────────────────────────
# Queries OSM road network per Tribe and computes total road length / area.
# Uses a 5km buffer around each Tribal land to capture nearby road context.

ox.settings.log_console = False
ox.settings.use_cache   = True

road_density_records = []
ROAD_BUFFER_KM = 5

for _, tribe in tribal_lands.iterrows():
    tb  = tribe.geometry.bounds
    buf = ROAD_BUFFER_KM / 111  # approximate degrees
    try:
        G = ox.graph_from_bbox(
            bbox=(tb[0]-buf, tb[1]-buf, tb[2]+buf, tb[3]+buf),
            network_type="drive",
            simplify=True,
        )
        edges = ox.graph_to_gdfs(G, nodes=False)
        total_road_km = edges["length"].sum() / 1000
        area_km2      = tribe["area_km2"]
        density       = total_road_km / area_km2 if area_km2 > 0 else 0
        road_density_records.append({
            "NAME":                  tribe["NAME"],
            "road_length_km":        round(total_road_km, 1),
            "road_density_km_km2":   round(density, 3),
        })
        print(f"  {tribe['NAME']}: {density:.3f} km/km²")
    except Exception as e:
        road_density_records.append({
            "NAME": tribe["NAME"],
            "road_length_km": np.nan,
            "road_density_km_km2": np.nan,
        })
        print(f"  {tribe['NAME']}: failed — {e}")

access_df = pd.DataFrame(road_density_records)
print("\nAccess component computed.")

In [ ]:
# ── Component 4: Population (Census ACS) ──────────────────────────────────────
# Requires CENSUS_API_KEY in .env — see configuration in Section 1.

if ACS_AVAILABLE:
    try:
        # Load ACS for two years to compute growth
        acs_2022 = loaders.load_census_acs_population(CENSUS_API_KEY, year=2022)
        acs_2017 = loaders.load_census_acs_population(CENSUS_API_KEY, year=2017)

        acs_2022 = acs_2022.rename(columns={"total_population": "pop_2022"})
        acs_2017 = acs_2017.rename(columns={"total_population": "pop_2017"})

        acs = acs_2022.merge(
            acs_2017[["state", "county", "pop_2017"]],
            on=["state", "county"],
        )
        acs["pop_growth_pct"] = (
            (acs["pop_2022"] - acs["pop_2017"]) / acs["pop_2017"] * 100
        ).round(2)

        # Spatial join ACS counties to Tribal lands to get local population growth
        # This requires loading county geometries — use Census TIGER counties
        # For now, flag as available and note the join approach
        print(f"ACS population data loaded: {len(acs):,} counties")
        print(f"Mean county pop growth 2017-2022: {acs['pop_growth_pct'].mean():.1f}%")
        population_df = acs
    except Exception as e:
        print(f"ACS loading failed: {e}")
        population_df = None
        ACS_AVAILABLE = False
else:
    population_df = None
    print(
        "CENSUS_API_KEY not set. Population component excluded from composite index.\n"
        "Get a free key at https://api.census.gov/data/key_signup.html\n"
        "Then set CENSUS_API_KEY=your_key in a .env file at repo root."
    )

---
## 4. Composite WUI Pressure Index

In [ ]:
# ── Assemble all components ────────────────────────────────────────────────────
scores = tribal_lands[["NAME", "area_km2", "edge_ratio"]].copy()
scores = scores.merge(proximity_df, on="NAME", how="left")
scores = scores.merge(access_df,   on="NAME", how="left")
if NLCD_AVAILABLE and expansion_df is not None:
    scores = scores.merge(expansion_df, on="NAME", how="left")

# ── Normalize each available component to 0–10 ────────────────────────────────
available_components = {}

# Proximity: closer = higher pressure (invert distance)
if "distance_to_urban_km" in scores.columns:
    max_dist = scores["distance_to_urban_km"].max()
    scores["proximity_score"] = (
        (max_dist - scores["distance_to_urban_km"]) / max_dist * 10
    ).clip(0, 10)
    available_components["proximity"] = WUI_COMPONENTS["proximity"]["weight"]

# Expansion: larger impervious change = higher pressure
if NLCD_AVAILABLE and "impervious_change" in scores.columns:
    max_change = scores["impervious_change"].clip(lower=0).max()
    if max_change > 0:
        scores["expansion_score"] = (
            scores["impervious_change"].clip(lower=0) / max_change * 10
        ).clip(0, 10)
        available_components["expansion"] = WUI_COMPONENTS["expansion"]["weight"]

# Access: higher road density = higher pressure
if "road_density_km_km2" in scores.columns:
    max_density = scores["road_density_km_km2"].max()
    if max_density > 0:
        scores["access_score"] = (
            scores["road_density_km_km2"] / max_density * 10
        ).clip(0, 10)
        available_components["access"] = WUI_COMPONENTS["access"]["weight"]

# Interface complexity: higher edge ratio = higher pressure
max_edge = scores["edge_ratio"].max()
if max_edge > 0:
    scores["complexity_score"] = (scores["edge_ratio"] / max_edge * 10).clip(0, 10)
    available_components["complexity"] = WUI_COMPONENTS["complexity"]["weight"]

# ── Renormalize weights for available components ───────────────────────────────
total_weight = sum(available_components.values())
norm_weights = {k: v / total_weight for k, v in available_components.items()}

print("Components included in composite index:")
for k, w in norm_weights.items():
    print(f"  {WUI_COMPONENTS[k]['name']}: {w:.1%} (original {WUI_COMPONENTS[k]['weight']:.1%})")
if len(norm_weights) < len(WUI_COMPONENTS):
    missing = set(WUI_COMPONENTS) - set(norm_weights)
    print(f"\nExcluded (data not available): {', '.join(missing)}")
    print("  Weights redistributed proportionally to available components.")

In [ ]:
# ── Compute composite index ────────────────────────────────────────────────────
scores["wui_pressure_index"] = sum(
    scores[f"{k}_score"] * w
    for k, w in norm_weights.items()
    if f"{k}_score" in scores.columns
) * 10  # scale to 0–100

scores["pressure_category"] = pd.cut(
    scores["wui_pressure_index"],
    bins=[0, 33, 66, 100],
    labels=["Low Pressure", "Moderate Pressure", "High Pressure"],
)
scores["pressure_rank"] = scores["wui_pressure_index"].rank(
    ascending=False, method="dense"
).astype(int)

print("WUI PRESSURE INDEX")
print("=" * 70)
print(
    scores[[
        "NAME", "wui_pressure_index", "pressure_category",
        "nearest_urban", "distance_to_urban_km",
    ]]
    .sort_values("wui_pressure_index", ascending=False)
    .to_string(index=False)
)
print(f"\nMean WUI Pressure: {scores['wui_pressure_index'].mean():.1f}")
print(scores["pressure_category"].value_counts().to_string())

---
## 5. Buffer Zones

In [ ]:
# ── Merge scores with geometry and build buffers ──────────────────────────────
tribal_wui = tribal_lands.merge(
    scores.drop(columns=["area_km2", "edge_ratio"], errors="ignore"),
    on="NAME",
)

tribal_proj = tribal_wui.to_crs("EPSG:5070")
buffers = {}
for dist_km in BUFFER_DISTANCES_KM:
    buff = tribal_proj.copy()
    buff["geometry"] = tribal_proj.buffer(dist_km * 1000)
    buff["buffer_km"] = dist_km
    buffers[dist_km] = buff.to_crs(constants.CRS_GEOGRAPHIC)

print("Buffer zones created:")
for d, b in buffers.items():
    area = b.to_crs("EPSG:5070").area.sum() / 1e6
    print(f"  {d} km: {area:,.0f} km² total")

---
## 6. Visualizations

In [ ]:
# ── Per-Tribe WUI pressure map ─────────────────────────────────────────────────
import contextily as ctx

WUI_CMAP = LinearSegmentedColormap.from_list(
    "wui", ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"], N=100
)

n_tribes = len(tribal_wui)
ncols, nrows = 2, (n_tribes + 1) // 2

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 5))
axes = axes.flatten()

for idx, (_, tribe_row) in enumerate(tribal_wui.iterrows()):
    ax   = axes[idx]
    name = tribe_row["NAME"]
    tb   = tribe_row.geometry.bounds
    buf  = 0.8

    clip = geo_utils.bbox_geodataframe(
        (tb[0]-buf, tb[1]-buf, tb[2]+buf, tb[3]+buf)
    ).geometry.iloc[0]

    # Urban areas near this Tribe
    local_urban = urban_areas[urban_areas.intersects(clip)].to_crs(3857)

    # Buffer rings
    for d_km in sorted(BUFFER_DISTANCES_KM, reverse=True):
        b_row = buffers[d_km][buffers[d_km]["NAME"] == name]
        if not b_row.empty:
            b_row.to_crs(3857).plot(
                ax=ax,
                facecolor="none",
                edgecolor=styles.SKY_BLUE,
                linewidth=0.8,
                linestyle="--",
                alpha=0.6,
            )

    # Tribal boundary colored by WUI pressure
    wui_val = tribe_row.get("wui_pressure_index", 0)
    rgba    = WUI_CMAP(wui_val / 100)
    hex_col = "#{:02x}{:02x}{:02x}".format(
        int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255)
    )
    gpd.GeoDataFrame(
        geometry=[tribe_row.geometry], crs=constants.CRS_GEOGRAPHIC
    ).to_crs(3857).plot(
        ax=ax, color=hex_col, edgecolor=styles.CHARCOAL,
        alpha=0.75, linewidth=1.5,
    )

    # Urban areas
    if not local_urban.empty:
        local_urban.plot(ax=ax, color="gray", alpha=0.25, edgecolor="black", linewidth=0.5)

    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.4)
    except Exception:
        pass

    cat = tribe_row.get("pressure_category", "N/A")
    ax.set_title(
        f"{name}\nWUI: {wui_val:.0f}/100 — {cat}",
        fontsize=9, fontweight="bold",
    )
    ax.set_axis_off()

for ax in axes[n_tribes:]:
    ax.set_visible(False)

fig.legend(
    handles=[
        mpatches.Patch(facecolor="#2ecc71", alpha=0.8, label="Low (0–33)"),
        mpatches.Patch(facecolor="#f1c40f", alpha=0.8, label="Moderate (33–66)"),
        mpatches.Patch(facecolor="#e74c3c", alpha=0.8, label="High (66–100)"),
        mpatches.Patch(facecolor="gray",    alpha=0.3, label="Urban area"),
    ],
    loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, 0.01),
)
plt.suptitle(
    "Tribal WUI Pressure Index\n"
    "Structural fire risk: proximity + access + interface complexity",
    fontsize=13, fontweight="bold",
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
charts.save_figure(fig, "outputs/figures/wui_pressure_map.png")
plt.show()

In [ ]:
# ── Component dashboard ───────────────────────────────────────────────────────
score_cols = [c for c in scores.columns if c.endswith("_score")]
n_comp     = len(score_cols) + 1  # +1 for composite
ncols_d    = min(3, n_comp)
nrows_d    = (n_comp + ncols_d - 1) // ncols_d

fig, axes = plt.subplots(nrows_d, ncols_d, figsize=(ncols_d * 6, nrows_d * 5))
axes      = np.array(axes).flatten()

comp_labels = {
    "proximity_score":  "Proximity Pressure",
    "expansion_score":  "Expansion Pressure (NLCD)",
    "access_score":     "Access Pressure (Road Density)",
    "population_score": "Population Pressure (ACS)",
    "complexity_score": "Interface Complexity",
}

for i, col in enumerate(score_cols):
    ax    = axes[i]
    data  = scores.sort_values(col, ascending=True)
    vals  = data[col]
    colors = [
        styles.EMBER_RED   if v >= 7 else
        styles.FIRE_ORANGE if v >= 4 else
        styles.SAGE_GREEN
        for v in vals
    ]
    ax.barh(data["NAME"], vals, color=colors, alpha=0.82)
    ax.set_title(comp_labels.get(col, col), fontsize=10, fontweight="bold")
    ax.set_xlabel("Score (0–10)")
    ax.set_xlim(0, 10)
    sns.despine(ax=ax)

# Composite index
ax   = axes[len(score_cols)]
data = scores.sort_values("wui_pressure_index", ascending=True)
vals = data["wui_pressure_index"]
colors = [
    styles.EMBER_RED   if v >= 66 else
    styles.FIRE_ORANGE if v >= 33 else
    styles.SAGE_GREEN
    for v in vals
]
ax.barh(data["NAME"], vals, color=colors, alpha=0.82)
ax.set_title("Composite WUI Pressure Index", fontsize=10, fontweight="bold")
ax.set_xlabel("Index (0–100)")
ax.set_xlim(0, 100)
sns.despine(ax=ax)

for ax in axes[n_comp:]:
    ax.set_visible(False)

plt.suptitle(
    "WUI Pressure Component Analysis",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/wui_component_dashboard.png")
plt.show()

In [ ]:
# ── Scatter: proximity vs. access (risk reframed) ─────────────────────────────
if "access_score" in scores.columns:
    fig, ax = plt.subplots(figsize=(10, 7))
    sc = ax.scatter(
        scores["distance_to_urban_km"],
        scores["road_density_km_km2"],
        c=scores["wui_pressure_index"],
        cmap="RdYlGn_r",
        s=scores["area_km2"].clip(upper=10000) / 10,
        alpha=0.75,
        edgecolors=styles.CHARCOAL,
        linewidth=0.8,
        vmin=0, vmax=100,
    )
    for _, row in scores.iterrows():
        ax.annotate(
            row["NAME"].split()[0],
            (row["distance_to_urban_km"], row["road_density_km_km2"]),
            xytext=(4, 4), textcoords="offset points", fontsize=8,
        )
    plt.colorbar(sc, ax=ax, label="WUI Pressure Index")
    ax.axvline(scores["distance_to_urban_km"].median(),
               color="gray", linestyle="--", alpha=0.5, label="Median")
    ax.axhline(scores["road_density_km_km2"].median(),
               color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Distance to Nearest Urban Area (km)", fontsize=11)
    ax.set_ylabel("Road Density (km/km²)", fontsize=11)
    ax.set_title(
        "Structural Fire Risk: Proximity vs. Access Pressure\n"
        "Closer ≠ higher pressure — roads matter too.",
        fontsize=12, fontweight="bold",
    )
    sns.despine(ax=ax)
    plt.tight_layout()
    charts.save_figure(fig, "outputs/figures/wui_structural_risk_scatter.png")
    plt.show()
else:
    print("Road density not available — scatter plot skipped.")

---
## 7. Export

In [ ]:
# ── Spatial exports ───────────────────────────────────────────────────────────
export_cols = [
    "NAME", "wui_pressure_index", "pressure_category",
    "nearest_urban", "distance_to_urban_km",
] + [c for c in scores.columns if c.endswith("_score")] + ["geometry"]

tribal_wui[
    [c for c in export_cols if c in tribal_wui.columns]
].assign(
    pressure_category=lambda df: df["pressure_category"].astype(str)
).to_file(
    constants.OUTPUTS_DIR / "tribal_wui_pressure.geojson",
    driver="GeoJSON",
)
print("Exported → outputs/tribal_wui_pressure.geojson")

for d_km, buf in buffers.items():
    buf.assign(
        pressure_category=lambda df: df["pressure_category"].astype(str)
    ).to_file(
        constants.OUTPUTS_DIR / f"tribal_wui_buffer_{d_km}km.geojson",
        driver="GeoJSON",
    )
    print(f"Exported → outputs/tribal_wui_buffer_{d_km}km.geojson")

# ── Tabular export ─────────────────────────────────────────────────────────────
scores.assign(
    pressure_category=lambda df: df["pressure_category"].astype(str)
).to_csv(
    constants.OUTPUTS_DIR / "wui_pressure_analysis.csv", index=False
)
print("Exported → outputs/wui_pressure_analysis.csv")

---
## 8. Summary & Findings

*(Fill in after running with real data.)*

Key questions to address in the narrative:
- Which Tribal lands score highest on the composite WUI Pressure Index,
  and which component drives that score most?
- Does proximity alone predict pressure, or do road density and interface
  complexity reveal a different picture? (This is the core 'risk reframing'
  argument — static distance ≠ structural pressure.)
- What are the limitations?
  - NLCD and ACS components may be absent; index weights are renormalized
    but results should be interpreted accordingly.
  - OSM road density reflects mapped roads; rural Tribal roads may be
    underrepresented in OSM coverage.
  - Census Urban Areas reflect 2020 delineation — fast-growing exurban
    areas may not yet be captured.
- Policy implication: high WUI pressure scores reflect factors outside
  Tribal control. This index supports arguments for proportional federal
  resources and regional coordination based on structural risk, not just
  historical fire acres.

---
## References

In [ ]:
print(generate_citations(["census_aiannh", "census_urban_areas", "nlcd", "census_acs"]))